# Sahasrabuddhe verification

This notebook reproduces the examples found at the end of the paper to verify that the new `LEV` and `Sahasrabuddhe` estimators work correctly.

## Example 1

### A1: Data and Information

The paper uses the well-known Mack triangle for its examples. This is the same as the `genins` sample triangle.

In [1]:
import chainladder as cl

genins = cl.load_sample("genins")
genins

,12,24,36,48,60,72,84,96,108,120
2001,"357,848","1,124,788","1,735,330","2,218,270","2,745,596","3,319,994","3,466,336","3,606,286","3,833,515","3,901,463"
2002,"352,118","1,236,139","2,170,033","3,353,322","3,799,067","4,120,063","4,647,867","4,914,039","5,339,085",
2003,"290,507","1,292,306","2,218,525","3,235,179","3,985,995","4,132,918","4,628,910","4,909,315",,
2004,"310,608","1,418,858","2,195,047","3,757,447","4,029,929","4,381,982","4,588,268",,,
2005,"443,160","1,136,350","2,128,333","2,897,821","3,402,672","3,873,311",,,,
2006,"396,132","1,333,217","2,180,715","2,985,752","3,691,712",,,,,
2007,"440,832","1,288,463","2,419,861","3,483,130",,,,,,
2008,"359,480","1,421,128","2,864,498",,,,,,,
2009,"376,686","1,363,294",,,,,,,,
2010,"344,014",,,,,,,,,


In [2]:
# Limits
triangle_limit = 1000000
basic_limit = 500000
policy_limit = 2000000

### B1: Exposure Period Trend Index

We'll need a 10 x 10 placeholder Triangle to visualize the trend indexes.

*2% EP (Exposure Period) Trend; 5% between EP 6 and 7*

In [3]:
# Create a placeholder triangle of ones.
ones = genins.copy()
ones.valuation_date = ones.valuation.max()
ones = (ones * 0 + 1).fillna(1)
ones

,12,24,36,48,60,72,84,96,108,120
2001,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000
2002,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000
2003,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000
2004,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000
2005,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000
2006,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000
2007,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000
2008,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000
2009,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000
2010,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000


Since chainladder trends to the latest exposure period by default, some manipulation is required to set the earliest exposure period to 1.

In [4]:
B1 = (
    1
    / cl
    .Trend(
        trends=[0.02, 0.05, 0.02],
        dates=[
            ("2006-12-31", None),  # 2% from the earliest origin up to EP 6
            ("2007-12-31", "2006-12-31"),  # 5% for the single EP 6 -> EP 7 step
            ("2010-12-31", "2007-12-31"),  # 2% from EP 7 through the latest origin
        ],
        axis="origin",
    )
    .fit(ones)
    .trend_
)
B1 = B1 / B1.iloc[0, 0, :1, :]

B1

,12,24,36,48,60,72,84,96,108,120
2001,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000
2002,1.0200,1.0200,1.0200,1.0200,1.0200,1.0200,1.0200,1.0200,1.0200,1.0200
2003,1.0404,1.0404,1.0404,1.0404,1.0404,1.0404,1.0404,1.0404,1.0404,1.0404
2004,1.0612,1.0612,1.0612,1.0612,1.0612,1.0612,1.0612,1.0612,1.0612,1.0612
2005,1.0824,1.0824,1.0824,1.0824,1.0824,1.0824,1.0824,1.0824,1.0824,1.0824
2006,1.1041,1.1041,1.1041,1.1041,1.1041,1.1041,1.1041,1.1041,1.1041,1.1041
2007,1.1593,1.1593,1.1593,1.1593,1.1593,1.1593,1.1593,1.1593,1.1593,1.1593
2008,1.1825,1.1825,1.1825,1.1825,1.1825,1.1825,1.1825,1.1825,1.1825,1.1825
2009,1.2061,1.2061,1.2061,1.2061,1.2061,1.2061,1.2061,1.2061,1.2061,1.2061
2010,1.2302,1.2302,1.2302,1.2302,1.2302,1.2302,1.2302,1.2302,1.2302,1.2302


### B2: Calendar Period Trend Index

*1% Calendar Period Trend; -5% between CP 2 and 3*

In [5]:
B2 = (
    1
    / cl
    .Trend(
        trends=[0.01, -0.05, 0.01],
        dates=[
            ("2002-12-31", None),  # 1% from the earliest valuation up to CP 2
            ("2003-12-31", "2002-12-31"),  # -5% for the single CP 2 -> CP 3 step
            (None, "2003-12-31"),  # 1% from CP 3 out to the last valuation
        ],
        axis="valuation",
    )
    .fit(ones)
    .trend_
)
B2 = B2 / B2.iloc[0, 0, :1, :1]

B2

,12,24,36,48,60,72,84,96,108,120
2001,1.0000,1.0100,0.9595,0.9691,0.9788,0.9886,0.9985,1.0084,1.0185,1.0287
2002,1.0100,0.9595,0.9691,0.9788,0.9886,0.9985,1.0084,1.0185,1.0287,1.0390
2003,0.9595,0.9691,0.9788,0.9886,0.9985,1.0084,1.0185,1.0287,1.0390,1.0494
2004,0.9691,0.9788,0.9886,0.9985,1.0084,1.0185,1.0287,1.0390,1.0494,1.0599
2005,0.9788,0.9886,0.9985,1.0084,1.0185,1.0287,1.0390,1.0494,1.0599,1.0705
2006,0.9886,0.9985,1.0084,1.0185,1.0287,1.0390,1.0494,1.0599,1.0705,1.0812
2007,0.9985,1.0084,1.0185,1.0287,1.0390,1.0494,1.0599,1.0705,1.0812,1.0920
2008,1.0084,1.0185,1.0287,1.0390,1.0494,1.0599,1.0705,1.0812,1.0920,1.1029
2009,1.0185,1.0287,1.0390,1.0494,1.0599,1.0705,1.0812,1.0920,1.1029,1.1139
2010,1.0287,1.0390,1.0494,1.0599,1.0705,1.0812,1.0920,1.1029,1.1139,1.1251


### B3: Combined Trend Index

`B1 * B2`

In [6]:
B3 = B1 * B2

B3

,12,24,36,48,60,72,84,96,108,120
2001,1.0000,1.0100,0.9595,0.9691,0.9788,0.9886,0.9985,1.0084,1.0185,1.0287
2002,1.0302,0.9787,0.9885,0.9984,1.0083,1.0184,1.0286,1.0389,1.0493,1.0598
2003,0.9983,1.0082,1.0183,1.0285,1.0388,1.0492,1.0597,1.0703,1.0810,1.0918
2004,1.0284,1.0387,1.0491,1.0596,1.0702,1.0809,1.0917,1.1026,1.1136,1.1248
2005,1.0595,1.0701,1.0808,1.0916,1.1025,1.1135,1.1246,1.1359,1.1473,1.1587
2006,1.0915,1.1024,1.1134,1.1245,1.1358,1.1471,1.1586,1.1702,1.1819,1.1937
2007,1.1575,1.1691,1.1808,1.1926,1.2045,1.2165,1.2287,1.2410,1.2534,1.2659
2008,1.1925,1.2044,1.2164,1.2286,1.2409,1.2533,1.2658,1.2785,1.2913,1.3042
2009,1.2285,1.2408,1.2532,1.2657,1.2783,1.2911,1.3040,1.3171,1.3303,1.3436
2010,1.2656,1.2782,1.2910,1.3039,1.3170,1.3301,1.3434,1.3569,1.3704,1.3841


Check the reconstruction against the exhibits in the paper.

In [7]:
import numpy as np

B1_published = np.tile(
    np.array([1.000, 1.020, 1.040, 1.061, 1.082, 1.104, 1.159, 1.182, 1.206, 1.230])[
        :, None
    ],
    (1, 10),
)

# B2 is constant along diagonals, so 19 calendar-period values determine it.
_cp = np.array([
    1.000,
    1.010,
    0.960,
    0.969,
    0.979,
    0.989,
    0.998,
    1.008,
    1.019,
    1.029,
    1.039,
    1.049,
    1.060,
    1.070,
    1.081,
    1.092,
    1.103,
    1.114,
    1.125,
])
_i, _j = np.indices((10, 10))
B2_published = _cp[_i + _j]

B3_published = np.array([
    [1.000, 1.010, 0.960, 0.969, 0.979, 0.989, 0.998, 1.008, 1.019, 1.029],
    [1.030, 0.979, 0.988, 0.998, 1.008, 1.018, 1.029, 1.039, 1.049, 1.060],
    [0.998, 1.008, 1.018, 1.029, 1.039, 1.049, 1.060, 1.070, 1.081, 1.092],
    [1.028, 1.039, 1.049, 1.060, 1.070, 1.081, 1.092, 1.103, 1.114, 1.125],
    [1.059, 1.070, 1.081, 1.092, 1.102, 1.114, 1.125, 1.136, 1.147, 1.159],
    [1.091, 1.102, 1.113, 1.125, 1.136, 1.147, 1.159, 1.170, 1.182, 1.194],
    [1.157, 1.169, 1.181, 1.193, 1.204, 1.217, 1.229, 1.241, 1.253, 1.266],
    [1.192, 1.204, 1.216, 1.229, 1.241, 1.253, 1.266, 1.278, 1.291, 1.304],
    [1.228, 1.241, 1.253, 1.266, 1.278, 1.291, 1.304, 1.317, 1.330, 1.344],
    [1.266, 1.278, 1.291, 1.304, 1.317, 1.330, 1.343, 1.357, 1.370, 1.384],
])

for name, computed, published in [
    ("B1", B1, B1_published),
    ("B2", B2, B2_published),
    ("B3", B3, B3_published),
]:
    deviation = np.abs(computed.values[0, 0] - published).max()
    assert deviation <= 0.0006, f"{name} deviates from the paper by {deviation:.6f}"
    print(f"{name}: matches the published exhibit (max deviation {deviation:.6f})")

B1: matches the published exhibit (max deviation 0.000471)
B2: matches the published exhibit (max deviation 0.000500)
B3: matches the published exhibit (max deviation 0.000500)


### T: the cost level index relative to the latest exposure period

Since chainladder's default trending behavior assumes the base period as the latest exposure period, we restate B3 to be relative to the 2010 cost level.

In [8]:
T = B3 / B3.iloc[0, 0, -1:, :]

T

,12,24,36,48,60,72,84,96,108,120
2001,0.7902,0.7902,0.7432,0.7432,0.7432,0.7432,0.7432,0.7432,0.7432,0.7432
2002,0.8140,0.7657,0.7657,0.7657,0.7657,0.7657,0.7657,0.7657,0.7657,0.7657
2003,0.7888,0.7888,0.7888,0.7888,0.7888,0.7888,0.7888,0.7888,0.7888,0.7888
2004,0.8126,0.8126,0.8126,0.8126,0.8126,0.8126,0.8126,0.8126,0.8126,0.8126
2005,0.8371,0.8371,0.8371,0.8371,0.8371,0.8371,0.8371,0.8371,0.8371,0.8371
2006,0.8624,0.8624,0.8624,0.8624,0.8624,0.8624,0.8624,0.8624,0.8624,0.8624
2007,0.9146,0.9146,0.9146,0.9146,0.9146,0.9146,0.9146,0.9146,0.9146,0.9146
2008,0.9422,0.9422,0.9422,0.9422,0.9422,0.9422,0.9422,0.9422,0.9422,0.9422
2009,0.9707,0.9707,0.9707,0.9707,0.9707,0.9707,0.9707,0.9707,0.9707,0.9707
2010,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000


### C1: Claims Size Model Parameters at Exposure Year 10 Cost Level

This paper assumes that mean claim sizes per development period can be supplied via an external model.

In [9]:
theta_10 = {
    12: 28138,
    24: 84242,
    36: 133998,
    48: 182460,
    60: 204649,
    72: 228245,
    84: 252830,
    96: 265063,
    108: 275707,
    120: 280000,
}

C1 = cl.LEV(means=theta_10).fit(genins).means_
C1

,12,24,36,48,60,72,84,96,108,120
2010,"28,138","84,242","133,998","182,460","204,649","228,245","252,830","265,063","275,707","280,000"


### C2: Claims Size Model Parameters

$C2_{i,j} = C1_j \, \dfrac{B3_{i,j}}{B3_{10,j}}$

Equation 3.1.  C1 holds one row of parameters, all at exposure year 10's cost
level. We now produce a triangle that restates the means from the claim size model to different cost levels.

In [10]:
lev = cl.LEV(
    means=theta_10,
    trend=T,
).fit(genins)

C2 = lev.means_
C2

,12,24,36,48,60,72,84,96,108,120
2001,"22,234","66,565","99,590","135,608","152,099","169,636","187,908","197,000","204,911","208,101"
2002,"22,905","64,501","102,598","139,703","156,692","174,759","193,583","202,949","211,099","214,386"
2003,"22,195","66,449","105,696","143,922","161,425","180,037","199,429","209,078","217,474","220,861"
2004,"22,865","68,456","108,888","148,269","166,300","185,474","205,452","215,393","224,042","227,531"
2005,"23,556","70,523","112,176","152,746","171,322","191,075","211,657","221,897","230,808","234,402"
2006,"24,267","72,653","115,564","157,359","176,496","196,846","218,049","228,599","237,778","241,481"
2007,"25,735","77,048","122,556","166,880","187,174","208,755","231,241","242,429","252,164","256,090"
2008,"26,512","79,375","126,257","171,919","192,826","215,059","238,224","249,750","259,779","263,824"
2009,"27,313","81,772","130,070","177,111","198,650","221,554","245,418","257,293","267,625","271,792"
2010,"28,138","84,242","133,998","182,460","204,649","228,245","252,830","265,063","275,707","280,000"


### C3: Unlimited Means

For an exponential model, the mean is the parameter, so C3 restates C2 without
changing it.  The paper prints it as its own exhibit because the general
method needs M(Phi) -- the unlimited first moment -- as a distinct quantity
from the parameters: for a lognormal or Pareto the two would differ.

In [11]:
C3 = cl.LEV(
    means=theta_10,
    trend=T,
    limit=np.inf,
).fit(genins).lev_
C3

,12,24,36,48,60,72,84,96,108,120
2001,"22,234","66,565","99,590","135,608","152,099","169,636","187,908","197,000","204,911","208,101"
2002,"22,905","64,501","102,598","139,703","156,692","174,759","193,583","202,949","211,099","214,386"
2003,"22,195","66,449","105,696","143,922","161,425","180,037","199,429","209,078","217,474","220,861"
2004,"22,865","68,456","108,888","148,269","166,300","185,474","205,452","215,393","224,042","227,531"
2005,"23,556","70,523","112,176","152,746","171,322","191,075","211,657","221,897","230,808","234,402"
2006,"24,267","72,653","115,564","157,359","176,496","196,846","218,049","228,599","237,778","241,481"
2007,"25,735","77,048","122,556","166,880","187,174","208,755","231,241","242,429","252,164","256,090"
2008,"26,512","79,375","126,257","171,919","192,826","215,059","238,224","249,750","259,779","263,824"
2009,"27,313","81,772","130,070","177,111","198,650","221,554","245,418","257,293","267,625","271,792"
2010,"28,138","84,242","133,998","182,460","204,649","228,245","252,830","265,063","275,707","280,000"


### C4: Limited Expected Values at Policy Limits

$\mathrm{LEV}(d) = \theta \, (1 - e^{-d / \theta})$, capping each claim at the policy limit.

In [12]:
C4 = cl.LEV(
    means=theta_10,
    trend=T,
    limit=policy_limit,
).fit(genins).lev_
C4

,12,24,36,48,60,72,84,96,108,120
2001,"22,234","66,565","99,590","135,608","152,099","169,635","187,904","196,992","204,899","208,087"
2002,"22,905","64,501","102,598","139,703","156,692","174,757","193,577","202,939","211,083","214,367"
2003,"22,195","66,449","105,696","143,922","161,424","180,034","199,420","209,064","217,452","220,835"
2004,"22,865","68,456","108,888","148,268","166,299","185,470","205,440","215,373","224,012","227,496"
2005,"23,556","70,523","112,176","152,746","171,320","191,070","211,640","221,870","230,768","234,356"
2006,"24,267","72,653","115,564","157,359","176,494","196,838","218,026","228,562","237,726","241,420"
2007,"25,735","77,048","122,556","166,878","187,169","208,740","231,200","242,366","252,073","255,987"
2008,"26,512","79,375","126,257","171,918","192,820","215,040","238,170","249,667","259,662","263,690"
2009,"27,313","81,772","130,070","177,109","198,641","221,527","245,347","257,184","267,473","271,619"
2010,"28,138","84,242","133,998","182,457","204,637","228,209","252,737","264,923","275,512","279,779"


In [13]:
cl.LEV(
    means=theta_10,
    trend=T,
    limit=policy_limit,
).fit(genins).means_

,12,24,36,48,60,72,84,96,108,120
2001,"22,234","66,565","99,590","135,608","152,099","169,636","187,908","197,000","204,911","208,101"
2002,"22,905","64,501","102,598","139,703","156,692","174,759","193,583","202,949","211,099","214,386"
2003,"22,195","66,449","105,696","143,922","161,425","180,037","199,429","209,078","217,474","220,861"
2004,"22,865","68,456","108,888","148,269","166,300","185,474","205,452","215,393","224,042","227,531"
2005,"23,556","70,523","112,176","152,746","171,322","191,075","211,657","221,897","230,808","234,402"
2006,"24,267","72,653","115,564","157,359","176,496","196,846","218,049","228,599","237,778","241,481"
2007,"25,735","77,048","122,556","166,880","187,174","208,755","231,241","242,429","252,164","256,090"
2008,"26,512","79,375","126,257","171,919","192,826","215,059","238,224","249,750","259,779","263,824"
2009,"27,313","81,772","130,070","177,111","198,650","221,554","245,418","257,293","267,625","271,792"
2010,"28,138","84,242","133,998","182,460","204,649","228,245","252,830","265,063","275,707","280,000"


### C5: Limited Expected Values at Limits of Data Triangle

In [14]:
C5 = cl.LEV(
    means=theta_10,
    trend=T,
    limit=triangle_limit,
).fit(genins).lev_
C5

,12,24,36,48,60,72,84,96,108,120
2001,"22,234","66,565","99,586","135,523","151,887","169,169","186,990","195,770","203,354","206,398"
2002,"22,905","64,501","102,592","139,594","156,427","174,187","192,478","201,479","209,249","212,366"
2003,"22,195","66,449","105,688","143,784","161,095","179,340","198,105","207,328","215,284","218,474"
2004,"22,865","68,456","108,877","148,094","165,893","184,629","203,871","213,318","221,460","224,723"
2005,"23,556","70,523","112,161","152,527","170,822","190,056","209,778","219,449","227,777","231,112"
2006,"24,267","72,653","115,544","157,086","175,885","195,621","215,826","225,720","234,233","237,640"
2007,"25,735","77,048","122,521","166,463","186,278","207,020","228,179","238,510","247,384","250,932"
2008,"26,512","79,375","126,211","171,407","191,748","213,003","234,644","245,194","254,248","257,866"
2009,"27,313","81,772","130,010","176,486","197,356","219,126","241,247","252,015","261,246","264,932"
2010,"28,138","84,241","133,921","181,700","203,104","225,390","247,987","258,969","268,375","272,128"


### C6: Limited Expected Values at Basic Limit

In [15]:
C6 = cl.LEV(
    means=theta_10,
    trend=T,
    limit=basic_limit,
).fit(genins).lev_
C6

,12,24,36,48,60,72,84,96,108,120
2001,"22,234","66,528","98,933","132,212","146,418","160,735","174,776","181,433","187,052","189,273"
2002,"22,905","64,473","101,813","135,805","150,247","164,762","178,956","185,674","191,337","193,574"
2003,"22,195","66,413","104,764","139,462","154,134","168,836","183,176","189,948","195,651","197,903"
2004,"22,865","68,410","107,785","143,181","158,075","172,957","187,431","194,254","199,993","202,256"
2005,"23,556","70,464","110,876","146,961","162,068","177,119","191,718","198,587","204,357","206,632"
2006,"24,267","72,578","114,037","150,799","166,111","181,322","196,035","202,944","208,742","211,026"
2007,"25,735","76,931","120,483","158,539","174,229","189,725","204,633","211,606","217,447","219,744"
2008,"26,512","79,229","123,851","162,538","178,404","194,029","209,019","216,018","221,873","224,175"
2009,"27,313","81,592","127,286","166,587","182,618","198,360","213,422","220,441","226,307","228,611"
2010,"28,138","84,019","130,788","170,682","186,869","202,717","217,839","224,873","230,745","233,050"


Check the transformer behavior.

In [16]:
lev_transformed = lev.transform(genins)

assert (lev_transformed == genins).all(), "transform leaves the values untouched"
assert np.allclose(
    np.nan_to_num(lev_transformed.means_.values),
    np.nan_to_num(C2.values),
), "the restated parameters ride along on the triangle"
assert np.allclose(
    np.nan_to_num(lev_transformed.lev_.values),
    np.nan_to_num(C3.values),
), "so does the layer expectation, here the unlimited mean"

print("transform attaches the fitted quantities and leaves the data alone")
print(f"  values unchanged : {(lev_transformed == genins).all()}")
print(f"  carries means_   : {hasattr(lev_transformed, 'means_')}")
print(f"  carries lev_     : {hasattr(lev_transformed, 'lev_')}")

transform attaches the fitted quantities and leaves the data alone
  values unchanged : True
  carries means_   : True
  carries lev_     : True


Check the C series against the exhibits as printed in the paper.

In [18]:
C2_published = np.array([
    [22233, 66564, 99590, 135608, 152099, 169636, 187908, 197000, 204911, 208101],
    [22905, 64501, 102598, 139703, 156693, 174759, 193583, 202949, 211099, 214386],
    [22195, 66449, 105696, 143922, 161425, 180037, 199429, 209078, 217475, 220861],
    [22865, 68455, 108888, 148268, 166300, 185474, 205452, 215392, 224042, 227531],
    [23555, 70523, 112177, 152746, 171322, 191075, 211657, 221897, 230808, 234402],
    [24267, 72653, 115564, 157359, 176496, 196846, 218049, 228598, 237779, 241481],
    [25735, 77048, 122556, 166879, 187174, 208755, 231241, 242429, 252164, 256090],
    [26512, 79375, 126257, 171919, 192827, 215059, 238224, 249750, 259780, 263824],
    [27313, 81772, 130070, 177111, 198650, 221554, 245418, 257292, 267625, 271792],
    [28138, 84242, 133998, 182460, 204649, 228245, 252830, 265063, 275707, 280000],
])

C4_published = np.array([
    [22233, 66564, 99590, 135607, 152099, 169635, 187904, 196992, 204899, 208087],
    [22905, 64501, 102598, 139703, 156692, 174757, 193577, 202938, 211083, 214367],
    [22195, 66449, 105696, 143922, 161424, 180034, 199420, 209063, 217453, 220835],
    [22865, 68455, 108888, 148268, 166299, 185470, 205440, 215372, 224013, 227496],
    [23555, 70523, 112177, 152746, 171321, 191070, 211640, 221870, 230769, 234356],
    [24267, 72653, 115564, 157358, 176494, 196838, 218026, 228562, 237726, 241420],
    [25735, 77048, 122556, 166878, 187170, 208740, 231200, 242365, 252074, 255987],
    [26512, 79375, 126257, 171917, 192821, 215039, 238170, 249667, 259662, 263690],
    [27313, 81772, 130070, 177109, 198642, 221527, 245348, 257184, 267473, 271619],
    [28138, 84242, 133998, 182456, 204638, 228209, 252737, 264922, 275512, 279779],
])

C5_published = np.array([
    [22233, 66564, 99586, 135522, 151887, 169169, 186990, 195770, 203355, 206398],
    [22905, 64501, 102592, 139594, 156428, 174187, 192478, 201479, 209249, 212366],
    [22195, 66449, 105688, 143784, 161096, 179340, 198105, 207328, 215285, 218474],
    [22865, 68455, 108877, 148094, 165893, 184629, 203871, 213318, 221461, 224723],
    [23555, 70523, 112161, 152527, 170822, 190056, 209778, 219448, 227777, 231112],
    [24267, 72652, 115544, 157085, 175885, 195621, 215826, 225719, 234233, 237640],
    [25735, 77048, 122521, 166462, 186279, 207020, 228179, 238510, 247385, 250932],
    [26512, 79375, 126211, 171407, 191748, 213003, 234644, 245194, 254248, 257866],
    [27313, 81772, 130010, 176486, 197356, 219126, 241247, 252014, 261246, 264932],
    [28138, 84241, 133921, 181699, 203105, 225390, 247987, 258969, 268375, 272128],
])

C6_published = np.array([
    [22233, 66528, 98933, 132211, 146418, 160735, 174776, 181433, 187052, 189273],
    [22905, 64473, 101813, 135805, 150248, 164762, 178957, 185674, 191337, 193574],
    [22195, 66413, 104764, 139462, 154134, 168836, 183176, 189948, 195652, 197903],
    [22865, 68409, 107785, 143181, 158075, 172956, 187431, 194253, 199993, 202256],
    [23555, 70464, 110876, 146960, 162068, 177119, 191718, 198586, 204358, 206632],
    [24267, 72578, 114037, 150798, 166111, 181322, 196035, 202944, 208743, 211026],
    [25735, 76931, 120483, 158539, 174229, 189725, 204633, 211606, 217447, 219744],
    [26512, 79229, 123851, 162538, 178404, 194029, 209019, 216018, 221873, 224175],
    [27313, 81591, 127286, 166587, 182619, 198360, 213422, 220441, 226307, 228611],
    [28138, 84019, 130788, 170682, 186869, 202716, 217839, 224873, 230745, 233050],
])

for name, computed, published in [
    ("C2", C2, C2_published),
    ("C3", C3, C2_published),
    ("C4", C4, C4_published),
    ("C5", C5, C5_published),
    ("C6", C6, C6_published),
]:
    deviation = np.abs(computed.values[0, 0] - published).max()
    assert deviation < 1.0, f"{name} deviates from the paper by {deviation:.2f}"
    print(f"{name}: matches the published exhibit (max deviation {deviation:.2f})")

C2: matches the published exhibit (max deviation 0.80)
C3: matches the published exhibit (max deviation 0.80)
C4: matches the published exhibit (max deviation 0.90)
C5: matches the published exhibit (max deviation 0.89)
C6: matches the published exhibit (max deviation 0.74)


### D: Calculation of Development Factors at Basic Limit

The Sahasrabuddhe estimator can be used to take a triangle at an observed limit and cost level, and restate it at another specified layer and cost level. Development factors can then be determined using the actuary's method of choice.

In [19]:
basic = cl.Sahasrabuddhe(
    means=theta_10,
    trend=T,
    observed_limit=triangle_limit,
    target_layer=(0, basic_limit),  # restate onto the basic limit
).fit(genins)

# D1: the triangle restated to exposure year 10 cost levels and basic limit.
D1 = basic.triangle_

development = cl.Development(average="volume").fit(D1)

D1

,12,24,36,48,60,72,84,96,108,120
2001,"452,881","1,419,731","2,279,040","2,793,771","3,377,949","3,978,376","4,038,185","4,142,392","4,349,867","4,405,265"
2002,"432,566","1,610,196","2,766,439","4,100,114","4,538,380","4,794,872","5,260,266","5,484,615","5,887,563",
2003,"368,296","1,634,013","2,745,402","3,840,400","4,623,710","4,671,635","5,090,015","5,324,756",,
2004,"382,236","1,741,436","2,636,785","4,330,557","4,539,484","4,811,269","4,902,614",,,
2005,"529,368","1,353,815","2,481,777","3,242,745","3,722,314","4,131,334",,,,
2006,"459,320","1,541,795","2,468,414","3,244,184","3,922,260",,,,,
2007,"481,990","1,405,036","2,583,136","3,571,424",,,,,,
2008,"381,520","1,504,277","2,968,365",,,,,,,
2009,"388,062","1,400,758",,,,,,,,
2010,"344,014",,,,,,,,,


### D2: Exposure Year 10 Incremental Basic Limit Development Factors

*Volume Weighted Averages*

In [20]:
D2 = development.ldf_
D2

,12-24,24-36,36-48,48-60,60-72,72-84,84-96,96-108,108-120
(All),3.5114,1.7141,1.3988,1.1472,1.0762,1.0567,1.0391,1.0634,1.0127


### D3: Exposure Year 10 Cumulative Development Factors

In [21]:
D3 = development.cdf_
D3

,12-Ult,24-Ult,36-Ult,48-Ult,60-Ult,72-Ult,84-Ult,96-Ult,108-Ult
(All),12.2919,3.5005,2.0422,1.4600,1.2727,1.1826,1.1191,1.0770,1.0127


Check the recreation of the D section against the paper.

One cell is excluded due to a typo.  The paper's
exhibit A1 has 359,840 at exposure period 8, development interval 1.  The
genins sample -- and the Taylor-Ashe triangle it comes from -- has 359,480.

In [22]:
TYPO = (7, 0)  # exposure period 8, development interval 1

D1_published = np.array(
    [
        [
            452881,
            1419731,
            2279040,
            2793772,
            3377947,
            3978376,
            4038185,
            4142394,
            4349865,
            4405265,
        ],
        [
            432566,
            1610197,
            2766439,
            4100116,
            4538378,
            4794873,
            5260266,
            5484617,
            5887561,
            None,
        ],
        [
            368296,
            1634013,
            2745402,
            3840401,
            4623708,
            4671636,
            5090015,
            5324759,
            None,
            None,
        ],
        [
            382236,
            1741436,
            2636785,
            4330559,
            4539482,
            4811270,
            4902613,
            None,
            None,
            None,
        ],
        [529368, 1353815, 2481777, 3242747, 3722312, 4131335, None, None, None, None],
        [459320, 1541795, 2468413, 3244186, 3922258, None, None, None, None, None],
        [481990, 1405037, 2583135, 3571425, None, None, None, None, None, None],
        [381903, 1504277, 2968365, None, None, None, None, None, None, None],
        [388062, 1400758, None, None, None, None, None, None, None, None],
        [344014, None, None, None, None, None, None, None, None, None],
    ],
    dtype="float64",
)

D2_published = np.array([3.511, 1.714, 1.399, 1.147, 1.076, 1.057, 1.039, 1.063, 1.013])
D3_published = np.array([
    12.291,
    3.501,
    2.042,
    1.460,
    1.273,
    1.183,
    1.119,
    1.077,
    1.013,
])

comparable = ~np.isnan(D1_published)
comparable[TYPO] = False
d1_dev = np.abs(D1.values[0, 0][comparable] - D1_published[comparable]).max()
assert d1_dev <= 3.0, f"D1 deviates from the paper by {d1_dev:.2f}"
print(
    f"D1: matches the published exhibit (max deviation {d1_dev:.2f}, 1 cell excluded)"
)

# The excluded cell: rerun it on the paper's own input and it reproduces the
# paper's own answer, which is what makes this a data typo rather than a bug.
ratio = C6.values[0, 0, -1, 0] / C5.values[0, 0][TYPO]
assert abs(359840 * ratio - D1_published[TYPO]) <= 1.0
assert abs(359480 * ratio - D1.values[0, 0][TYPO]) <= 1.0
print(
    f"    the excluded cell reconciles: paper's A1 359,840 -> {359840 * ratio:,.0f} "
    f"(paper prints {D1_published[TYPO]:,.0f}); "
    f"genins' 359,480 -> {359480 * ratio:,.0f}"
)

d2_dev = np.abs(D2.values[0, 0, 0] - D2_published).max()
assert d2_dev <= 0.0005, f"D2 deviates from the paper by {d2_dev:.6f}"
print(f"D2: matches the published exhibit (max deviation {d2_dev:.6f})")

d3_dev = np.abs(D3.values[0, 0, 0] - D3_published).max()
assert d3_dev <= 0.002, f"D3 deviates from the paper by {d3_dev:.6f}"
print(f"D3: matches the published exhibit (max deviation {d3_dev:.6f})")

D1: matches the published exhibit (max deviation 2.69, 1 cell excluded)
    the excluded cell reconciles: paper's A1 359,840 -> 381,903 (paper prints 381,903); genins' 359,480 -> 381,520
D2: matches the published exhibit (max deviation 0.000410)
D3: matches the published exhibit (max deviation 0.000862)


### E: Calculation of Development Factors by Layer

### E1: Basic Limit

In [23]:
def layer(target_layer):
    return cl.Sahasrabuddhe(
        means=theta_10,
        trend=T,
        basic_limit=basic_limit,
        target_layer=target_layer,
    ).fit(development)


E1 = layer(target_layer=(0, basic_limit)).full_cdf_
E1

,12,24,36,48,60,72,84,96,108,120
2001,12.6340,3.5904,2.1927,1.5308,1.3192,1.2113,1.1328,1.0841,1.0146,1.0000
2002,12.5423,3.7891,2.1790,1.5242,1.3148,1.2085,1.1315,1.0834,1.0144,1.0000
2003,13.2330,3.7606,2.1650,1.5174,1.3103,1.2057,1.1302,1.0827,1.0143,1.0000
2004,13.1277,3.7312,2.1506,1.5105,1.3057,1.2029,1.1288,1.0820,1.0141,1.0000
2005,13.0185,3.7008,2.1359,1.5035,1.3011,1.2000,1.1274,1.0813,1.0139,1.0000
2006,12.9057,3.6694,2.1209,1.4964,1.2964,1.1971,1.1261,1.0805,1.0137,1.0000
2007,12.6721,3.6048,2.0903,1.4821,1.2871,1.1914,1.1233,1.0791,1.0133,1.0000
2008,12.5487,3.5708,2.0745,1.4748,1.2823,1.1884,1.1219,1.0784,1.0131,1.0000
2009,12.4219,3.5360,2.0585,1.4674,1.2775,1.1855,1.1205,1.0777,1.0129,1.0000
2010,12.2919,3.5005,2.0422,1.4600,1.2727,1.1826,1.1191,1.0770,1.0127,1.0000


### E1 as a development pattern

In [24]:
layer(target_layer=(0, basic_limit)).cdf_

,12-Ult,24-Ult,36-Ult,48-Ult,60-Ult,72-Ult,84-Ult,96-Ult,108-Ult,120-Ult
(All),12.2919,3.5360,2.0745,1.4821,1.2964,1.2000,1.1288,1.0827,1.0144,1.0000


### E2: Basic Limit to Policy Limit

In [25]:
E2 = layer(target_layer=(basic_limit, policy_limit)).full_cdf_
E2.to_frame(origin_as_datetime=False).style.format("{:,.3f}", na_rep="")

,12,24,36,48,60,72,84,96,108,120
2001,"7,338,449,089.189",652.392,32.803,5.924,3.380,2.175,1.499,1.257,1.057,1.000
2002,"4,072,028,337.971",946.201,30.374,5.704,3.293,2.140,1.488,1.252,1.056,1.000
2003,"9,317,588,932.690",807.040,28.188,5.498,3.210,2.107,1.477,1.247,1.055,1.000
2004,"5,142,956,628.735",691.533,26.215,5.305,3.132,2.075,1.466,1.242,1.054,1.000
2005,"2,888,454,818.061",595.215,24.431,5.124,3.058,2.044,1.456,1.237,1.053,1.000
2006,"1,649,821,704.129",514.540,22.814,4.954,2.987,2.015,1.446,1.233,1.052,1.000
2007,"572,641,375.698",390.696,20.042,4.647,2.858,1.961,1.427,1.224,1.050,1.000
2008,"342,891,492.826",341.857,18.820,4.505,2.797,1.935,1.418,1.220,1.049,1.000
2009,"208,409,717.503",300.267,17.704,4.371,2.739,1.910,1.409,1.216,1.048,1.000
2010,"128,520,235.973",264.714,16.682,4.244,2.684,1.885,1.401,1.212,1.047,1.000


### E3: Policy Limit to Unlimited

In [26]:
E3 = layer(target_layer=(policy_limit, np.inf)).full_cdf_
E3.to_frame(origin_as_datetime=False).style.format("{:,.3f}", na_rep="")

,12,24,36,48,60,72,84,96,108,120
2001,,"2,956,805,468.668","84,540.807",279.494,48.058,11.155,3.254,1.887,1.183,1.000
2002,,"10,899,606,407.922","62,194.144",240.416,43.323,10.464,3.157,1.857,1.178,1.000
2003,,"5,778,339,392.258","46,167.967",207.717,39.173,9.835,3.066,1.829,1.173,1.000
2004,,"3,116,347,385.523","34,572.085",180.236,35.525,9.260,2.979,1.801,1.168,1.000
2005,,"1,712,071,696.674","26,109.152",157.042,32.310,8.735,2.898,1.775,1.163,1.000
2006,,"957,330,170.420","19,880.824",137.387,29.467,8.253,2.821,1.751,1.158,1.000
2007,,"319,329,226.902","11,880.984",106.721,24.762,7.415,2.681,1.705,1.150,1.000
2008,,"187,586,741.704","9,258.016",94.427,22.760,7.040,2.616,1.683,1.146,1.000
2009,,"111,925,552.619","7,267.068",83.850,20.972,6.694,2.555,1.662,1.142,1.000
2010,,"67,800,115.160","5,744.910",74.718,19.371,6.374,2.496,1.642,1.138,1.000


### E4: Limit of Data in Triangle

In [27]:
E4 = layer(target_layer=(0, triangle_limit)).full_cdf_
E4

,12,24,36,48,60,72,84,96,108,120
2001,13.7771,3.9132,2.3754,1.6285,1.3867,1.2550,1.1546,1.0956,1.0177,1.0000
2002,13.7599,4.1551,2.3724,1.6267,1.3854,1.2541,1.1541,1.0953,1.0177,1.0000
2003,14.6086,4.1493,2.3692,1.6248,1.3840,1.2531,1.1536,1.0950,1.0176,1.0000
2004,14.5859,4.1429,2.3656,1.6226,1.3824,1.2520,1.1531,1.0947,1.0175,1.0000
2005,14.5609,4.1358,2.3616,1.6202,1.3807,1.2508,1.1524,1.0944,1.0174,1.0000
2006,14.5333,4.1279,2.3572,1.6176,1.3788,1.2496,1.1518,1.0940,1.0173,1.0000
2007,14.4707,4.1102,2.3473,1.6119,1.3747,1.2468,1.1504,1.0933,1.0171,1.0000
2008,14.4346,4.0999,2.3416,1.6087,1.3724,1.2453,1.1496,1.0929,1.0170,1.0000
2009,14.3954,4.0888,2.3355,1.6052,1.3699,1.2437,1.1488,1.0924,1.0169,1.0000
2010,14.3529,4.0767,2.3289,1.6015,1.3673,1.2419,1.1479,1.0920,1.0167,1.0000


### E5: Unadjusted Incremental Development Factors at Limits of Data Triangle

*Volume Weighted Averages*

In [28]:
unadjusted = cl.Development(average="volume").fit(genins)

E5 = unadjusted.ldf_
E5

,12-24,24-36,36-48,48-60,60-72,72-84,84-96,96-108,108-120
(All),3.4906,1.7473,1.4574,1.1739,1.1038,1.0863,1.0539,1.0766,1.0177


### E6: Unadjusted Cumulative Development Factors

In [29]:
E6 = unadjusted.cdf_
E6

,12-Ult,24-Ult,36-Ult,48-Ult,60-Ult,72-Ult,84-Ult,96-Ult,108-Ult
(All),14.4466,4.1387,2.3686,1.6252,1.3845,1.2543,1.1547,1.0956,1.0177


### E7: Differences

In [30]:
import pandas as pd

last_diagonal = np.array([E4.values[0, 0, 10 - 1 - j, j] for j in range(10)])
labels = list(E6.development) + ["120-Ult"]

E7 = pd.Series(
    (np.append(E6.values[0, 0, 0], 1.0) / last_diagonal - 1) * 100, index=labels
)
E7.round(1).to_frame("difference %").T

,12-Ult,24-Ult,36-Ult,48-Ult,60-Ult,72-Ult,84-Ult,96-Ult,108-Ult,120-Ult
difference %,0.7,1.2,1.2,0.8,0.4,0.3,0.1,0.1,0.0,0.0


Check the E series against the exhibits as printed in the paper.

E2 and E3 are compared only where the paper prints a number: it writes "very
large" wherever the layer has essentially no expected loss at that age, and
those cells are non-finite or astronomically large here for the same reason.

In [31]:
VERY_LARGE = np.nan

E1_published = np.array(
    [
        [12.633, 3.590, 2.193, 1.531, 1.319, 1.211, 1.133, 1.084, 1.015, 1.000],
        [12.541, 3.789, 2.179, 1.524, 1.315, 1.209, 1.132, 1.083, 1.014, None],
        [13.232, 3.761, 2.165, 1.517, 1.310, 1.206, 1.130, 1.083, None, None],
        [13.126, 3.731, 2.151, 1.510, 1.306, 1.203, 1.129, None, None, None],
        [13.017, 3.701, 2.136, 1.503, 1.301, 1.200, None, None, None, None],
        [12.904, 3.669, 2.121, 1.496, 1.296, None, None, None, None, None],
        [12.671, 3.605, 2.090, 1.482, None, None, None, None, None, None],
        [12.547, 3.571, 2.074, None, None, None, None, None, None, None],
        [12.421, 3.536, None, None, None, None, None, None, None, None],
        [12.291, None, None, None, None, None, None, None, None, None],
    ],
    dtype="float64",
)

E2_published = np.array(
    [
        [VERY_LARGE, 652.420, 32.802, 5.924, 3.380, 2.175, 1.499, 1.257, 1.057, 1.000],
        [VERY_LARGE, 946.242, 30.374, 5.704, 3.293, 2.140, 1.488, 1.252, 1.056, None],
        [VERY_LARGE, 807.075, 28.187, 5.498, 3.210, 2.107, 1.477, 1.247, None, None],
        [VERY_LARGE, 691.561, 26.215, 5.305, 3.132, 2.075, 1.466, None, None, None],
        [VERY_LARGE, 595.239, 24.431, 5.124, 3.058, 2.044, None, None, None, None],
        [VERY_LARGE, 514.560, 22.814, 4.954, 2.987, None, None, None, None, None],
        [VERY_LARGE, 390.710, 20.042, 4.647, None, None, None, None, None, None],
        [VERY_LARGE, 341.869, 18.820, None, None, None, None, None, None, None],
        [VERY_LARGE, 300.278, None, None, None, None, None, None, None, None],
        [VERY_LARGE, None, None, None, None, None, None, None, None, None],
    ],
    dtype="float64",
)

E3_published = np.array(
    [
        [
            VERY_LARGE,
            VERY_LARGE,
            84538.278,
            279.503,
            48.056,
            11.155,
            3.254,
            1.887,
            1.183,
            1.000,
        ],
        [
            VERY_LARGE,
            VERY_LARGE,
            62192.336,
            240.423,
            43.321,
            10.464,
            3.157,
            1.857,
            1.178,
            None,
        ],
        [
            VERY_LARGE,
            VERY_LARGE,
            46166.664,
            207.723,
            39.172,
            9.835,
            3.066,
            1.829,
            None,
            None,
        ],
        [
            VERY_LARGE,
            VERY_LARGE,
            34571.138,
            180.241,
            35.524,
            9.261,
            2.979,
            None,
            None,
            None,
        ],
        [
            VERY_LARGE,
            VERY_LARGE,
            26108.458,
            157.047,
            32.309,
            8.735,
            None,
            None,
            None,
            None,
        ],
        [
            VERY_LARGE,
            VERY_LARGE,
            19880.311,
            137.391,
            29.466,
            None,
            None,
            None,
            None,
            None,
        ],
        [
            VERY_LARGE,
            VERY_LARGE,
            11880.695,
            106.724,
            None,
            None,
            None,
            None,
            None,
            None,
        ],
        [VERY_LARGE, VERY_LARGE, 9257.797, None, None, None, None, None, None, None],
        [VERY_LARGE, VERY_LARGE, None, None, None, None, None, None, None, None],
        [VERY_LARGE, None, None, None, None, None, None, None, None, None],
    ],
    dtype="float64",
)

E4_published = np.array(
    [
        [13.776, 3.913, 2.375, 1.629, 1.387, 1.255, 1.155, 1.096, 1.018, 1.000],
        [13.759, 4.155, 2.372, 1.627, 1.385, 1.254, 1.154, 1.095, 1.018, None],
        [14.607, 4.149, 2.369, 1.625, 1.384, 1.253, 1.154, 1.095, None, None],
        [14.584, 4.143, 2.366, 1.623, 1.382, 1.252, 1.153, None, None, None],
        [14.559, 4.136, 2.362, 1.620, 1.381, 1.251, None, None, None, None],
        [14.532, 4.128, 2.357, 1.618, 1.379, None, None, None, None, None],
        [14.469, 4.110, 2.347, 1.612, None, None, None, None, None, None],
        [14.433, 4.100, 2.342, None, None, None, None, None, None, None],
        [14.394, 4.089, None, None, None, None, None, None, None, None],
        [14.352, None, None, None, None, None, None, None, None, None],
    ],
    dtype="float64",
)

E5_published = np.array([3.490, 1.747, 1.457, 1.174, 1.104, 1.086, 1.054, 1.077, 1.018])
E6_published = np.array([
    14.445,
    4.139,
    2.369,
    1.625,
    1.384,
    1.254,
    1.155,
    1.096,
    1.018,
])
E7_published = np.array([0.7, 1.2, 1.2, 0.8, 0.4, 0.3, 0.1, 0.1, 0.0, 0.0])

for name, computed, published, tol in [
    ("E1", E1.values[0, 0], E1_published, 0.002),
    ("E2", E2.values[0, 0], E2_published, 0.002),
    ("E3", E3.values[0, 0], E3_published, 0.010),
    ("E4", E4.values[0, 0], E4_published, 0.002),
]:
    comparable = np.isfinite(published)
    # E3's largest printed factor carries five significant figures, so compare
    # it relatively rather than absolutely.
    deviation = np.abs(computed[comparable] / published[comparable] - 1).max()
    assert deviation <= tol, f"{name} deviates from the paper by {deviation:.5f}"
    print(
        f"{name}: matches the published exhibit "
        f"(max relative deviation {deviation:.5f}, {comparable.sum()} cells)"
    )

E1: matches the published exhibit (max relative deviation 0.00043, 55 cells)
E2: matches the published exhibit (max relative deviation 0.00038, 45 cells)
E3: matches the published exhibit (max relative deviation 0.00039, 36 cells)
E4: matches the published exhibit (max relative deviation 0.00039, 55 cells)


E5 and E6 are fitted on the raw triangle, so the A1 typo found in the D
series reaches them too -- and only through the 12-24 factor, whose
denominator includes exposure period 8 at age 12.  The first factor is
therefore checked by reconciliation rather than by tolerance.

In [32]:
raw = genins.values[0, 0]
paper_ldf_12_24 = np.nansum(raw[:9, 1]) / (np.nansum(raw[:9, 0]) + (359840 - 359480))
assert abs(paper_ldf_12_24 - E5_published[0]) <= 0.0005
print(
    f"E5: first factor reconciles to the paper's A1 "
    f"({paper_ldf_12_24:.5f} vs published {E5_published[0]:.3f}); "
    f"genins gives {E5.values[0, 0, 0, 0]:.5f}"
)

for name, computed, published, tol, skip in [
    ("E5", E5.values[0, 0, 0], E5_published, 0.0005, 1),
    ("E6", E6.values[0, 0, 0], E6_published, 0.002, 0),
    ("E7", E7.values, E7_published, 0.05, 0),
]:
    deviation = np.abs(computed[skip:] - published[skip:]).max()
    assert deviation <= tol, f"{name} deviates from the paper by {deviation:.5f}"
    note = " (first factor excluded)" if skip else ""
    print(
        f"{name}: matches the published exhibit (max deviation {deviation:.5f}){note}"
    )

E5: first factor reconciles to the paper's A1 (3.49023 vs published 3.490); genins gives 3.49061
E5: matches the published exhibit (max deviation 0.00044) (first factor excluded)
E6: matches the published exhibit (max deviation 0.00158)
E7: matches the published exhibit (max deviation 0.04912)
